In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [ ]:
x_train.shape

In [ ]:
d = {
    "casual": {
        0: 1,
        1: 1,
        2: 1,
        3: 0, 
        4: 0,
        5: 1,
        6: 0,
        7: 1,
        8: 0,
        9: 0
    }, 
    "accessory": {
        0: 0,
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: 1,
        6: 0,
        7: 1,
        8: 1,
        9: 1
    },
    "top": {
        0: 1,
        1: 0,
        2: 1,
        3: 0,
        4: 0,
        5: 0, 
        6: 1,
        7: 0,
        8: 1,
        9: 0
    },
    "seasonal": {
        0: 1,
        1: 0,
        2: 1,
        3: 0,
        4: 1,
        5: 1,
        6: 0,
        7: 0,
        8: 0,
        9: 1
    },
    "technical": {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
        5: 0, 
        6: 1,
        7: 0,
        8: 0,
        9: 0
    }
}

In [ ]:
count = {i: 0 for i in range(10)}
for k in d:
    for i in range(10):
        if d[k][i] == 1:
            count[i] += 1
count

In [ ]:
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
from sklearn.manifold import TSNE

In [ ]:
def build_df(x, y):
    x = x.reshape(x.shape[0], -1)
    df = pd.DataFrame(x)
    df.columns = [f"pixel{i}" for i in range(1,len(df.columns)+1)]
    df["target"] = y
    df["task1"] = df["target"].apply(lambda x : d["casual"][x])
    df["task2"] = df["target"].apply(lambda x : d["accessory"][x])
    df["task3"] = df["target"].apply(lambda x : d["top"][x])
    df["task4"] = df["target"].apply(lambda x : d["seasonal"][x])
    df["task5"] = df["target"].apply(lambda x : d["technical"][x])
    for i in range(1, 785):
        df[f"pixel{i}"] = df[f"pixel{i}"] / 255
    df = df.sample(frac=1).reset_index(drop=True)
    return df

In [ ]:
train_df = build_df(x_train, y_train)
test_df = build_df(x_test, y_test)

In [ ]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

In [ ]:
train_df.to_csv(os.path.join(root, "datasets", "fashion_mnist_train.csv"), index=False)
test_df.to_csv(os.path.join(root, "datasets", "fashion_mnist_test.csv"), index=False)

In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import umap

In [ ]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

train_df = pd.read_csv(os.path.join(root, "datasets", "fashion_mnist_train.csv"))
test_df = pd.read_csv(os.path.join(root, "datasets", "fashion_mnist_test.csv"))

In [ ]:
umap_model = umap.UMAP(n_components=50)
umap_model.fit(train_df.iloc[:,:784].values)

In [ ]:
X_train_red = umap_model.transform(train_df.iloc[:,:784].values)
X_test_red = umap_model.transform(test_df.iloc[:,:784].values)

In [ ]:
def red(X, df):
    df_red = pd.DataFrame(X, columns=[f"feat{i}" for i in range(X.shape[1])])
    df_red["target"] = list(df["target"])
    for i in range(1,6):
        df_red[f"task{i}"] = list(df[f"task{i}"])
    return df_red

In [ ]:
df_train_red = red(X_train_red, train_df)
df_test_red = red(X_test_red, test_df)
df_train_red.to_csv(os.path.join(root, "datasets", "fashion_mnist_red50_train.csv"), index=False)
df_test_red.to_csv(os.path.join(root, "datasets", "fashion_mnist_red50_test.csv"), index=False)

In [1]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import pickle
import itertools
import numpy as np

root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

df_train_red = pd.read_csv(os.path.join(root, "datasets", "fashion_mnist_red50_train.csv"))
df_test_red = pd.read_csv(os.path.join(root, "datasets", "fashion_mnist_red50_test.csv"))

In [2]:
from dataset_utils import *

In [3]:
set1=(0, 1, 2, 5, 7)
set2=(3, 4, 6, 8, 9)

In [4]:
dataset = "fashion_mnist_red50"

In [ ]:
exps_train = make_exp_incremental_train(root, dataset, df_train_red, set1, set2)
exps_test = make_exp_incremental_test(root, dataset, df_test_red)

In [ ]:
exps_sml_train = make_exp_sml_train(root, dataset, df_train_red)
exps_sml_test = make_exp_sml_test(root, dataset, df_test_red)

In [ ]:
check_distr(exps_train)
check_distr(exps_test)
check_distr(exps_sml_train)
check_distr(exps_sml_test)

In [5]:
exps_sml_train_3 = make_exp_sml_train(root, dataset, df_train_red, n_split=3, suffix="_3")
exps_sml_test_3 = make_exp_sml_test(root, dataset, df_test_red, n_split=3, suffix="_3")

(4, 3, 1) ACCEPTED
(5, 3, 1) ACCEPTED
(3, 1, 5) ACCEPTED
(3, 1, 2) ACCEPTED
(1, 2, 5) ACCEPTED
(5, 2, 4) ACCEPTED
(1, 5, 2) ACCEPTED
(5, 1, 2) ACCEPTED
(1, 3, 4) ACCEPTED
(1, 4, 5) ACCEPTED


In [5]:
exps_sml_train_2 = make_exp_sml_train(
    root,
    dataset,
    df_train_red,
    tasks_perc=[0.3, 0.7],
    suffix="_2"
)
exps_sml_test_2 = make_exp_sml_test(root, dataset, df_test_red, n_split=2, suffix="_2")

(4, 1) ACCEPTED
(3, 4) ACCEPTED
(4, 1) REJECTED
(5, 3) ACCEPTED
(1, 5) ACCEPTED
(1, 3) ACCEPTED
(4, 2) ACCEPTED
(2, 1) ACCEPTED
(2, 1) ACCEPTED
(1, 3) ACCEPTED
(3, 2) ACCEPTED
